# 单因子研究模板

本 notebook 是因子研究的标准工作流模板。
覆盖从数据准备到因子入库的完整流程。

## 工作流
1. 环境设置
2. 数据准备（合成 / 真实）
3. 因子计算（手写 / FactorEngine）
4. 因子评价（FactorAnalyzer 一键报告）
5. 深入分析（IC 衰减、分层回测、稳定性）
6. 因子代码化模板
7. 存储入库

## 1. 环境设置

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 因子研究框架
from factor_research.evaluation.analyzer import FactorAnalyzer
from factor_research.evaluation.ic import ic_analysis, ic_series, ic_summary
from factor_research.evaluation.quantile import quantile_backtest
from factor_research.evaluation.tail import tail_analysis
from factor_research.evaluation.stability import stability_analysis
from factor_research.evaluation.report import format_report_text, plot_report
from factor_research.store.factor_store import FactorStore
from factor_research.core.types import FactorMeta, FactorType, DataRequest, DataType

plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)
print("环境加载完成")

## 2. 数据准备

### 方式 A: 合成数据（无需数据库，随时可用）

In [ ]:
# === 合成数据 ===
n = 2000
symbols = ["BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT", "DOGE/USDT"]
index = pd.date_range("2024-01-01", periods=n, freq="1min", tz="UTC")

# 生成带有共同因子的价格
common_signal = np.random.randn(n) * 0.001
price_data = {}
for sym in symbols:
    noise = np.random.randn(n) * 0.0005
    returns = common_signal + noise
    prices = 100 * np.exp(np.cumsum(returns))
    price_data[sym] = prices

price_panel = pd.DataFrame(price_data, index=index)
print(f"价格面板: {price_panel.shape}")
price_panel.head()

In [ ]:
# === 方式 B: 真实数据（需要已采集数据）===
# 取消注释以下代码使用真实数据:
#
# from data_infra.data.reader import DataReader
# reader = DataReader()
# ohlcv_btc = reader.get_ohlcv("BTC/USDT", timeframe="1m", limit=2000)
# ohlcv_eth = reader.get_ohlcv("ETH/USDT", timeframe="1m", limit=2000)
# # ... 其他标的
#
# # 构建价格面板
# price_panel = pd.DataFrame({
#     "BTC/USDT": ohlcv_btc.set_index("timestamp")["close"],
#     "ETH/USDT": ohlcv_eth.set_index("timestamp")["close"],
# })
print("如需使用真实数据，请取消上方注释")

## 3. 因子计算

### 方式 A: 手写因子逻辑（探索阶段）

在确定因子逻辑之前，先在 notebook 中快速验证想法。

In [ ]:
# 示例: 5 分钟动量因子
lookback = 5
factor_panel = price_panel.pct_change(lookback)

# 去掉前 lookback 行的 NaN
factor_panel = factor_panel.iloc[lookback:]
print(f"因子面板: {factor_panel.shape}")
factor_panel.head()

In [ ]:
# === 方式 B: 通过 FactorEngine 计算（代码化后）===
# 取消注释以下代码:
#
# from factor_research.core.engine import FactorEngine
# from factor_research.core.registry import FactorRegistry
# import factor_research.factors.momentum.returns  # 触发注册
#
# reader = DataReader()
# store = FactorStore()
# engine = FactorEngine(reader=reader, store=store)
# factor_panel = engine.compute_factor(
#     "returns_5m",
#     symbols=symbols,
#     save=False,
# )
print("如需通过 FactorEngine 计算，请取消上方注释")

## 4. 因子评价 — FactorAnalyzer 一键报告

In [ ]:
analyzer = FactorAnalyzer(factor_panel, price_panel)

# 文本报告
print(analyzer.summary_text(factor_name="momentum_5m"))

In [ ]:
# 可视化报告
report = analyzer.full_report(horizons=[1, 5, 10, 30, 60])
figures = plot_report(report, factor_name="momentum_5m")

for name, fig in figures.items():
    fig.set_size_inches(10, 5)
    plt.figure(fig.number)
    plt.show()

## 5. 深入分析

### 5.1 IC 衰减解读

IC 随 horizon 的衰减速度反映因子的持久性:
- 快速衰减: 短期信号，需要高频换仓
- 缓慢衰减: 信号持久，适合低频策略

In [ ]:
ic_report = report["ic"]
print("IC 衰减:")
print(ic_report["ic_decay"])

### 5.2 分层回测解读

单调性是因子有效性的关键指标:
- 单调性 > 0.8: 因子分层效果好
- 多空收益 Sharpe > 1: 因子有显著的赚钱能力

In [ ]:
quantile_report = report["quantile"]
print(f"分组收益: {quantile_report['group_returns']}")
print(f"多空收益: {quantile_report['long_short_return']:.4f}")
print(f"多空 Sharpe: {quantile_report['long_short_sharpe']:.4f}")
print(f"单调性: {quantile_report['monotonicity']:.4f}")

### 5.3 稳定性诊断

检查因子在不同 regime 下是否稳定有效。

In [ ]:
stab = stability_analysis(factor_panel, price_panel, horizon=1)
print("Regime IC:")
print(f"  上涨: {stab['regime_ic']['trend'].get('uptrend', {})}")
print(f"  下跌: {stab['regime_ic']['trend'].get('downtrend', {})}")
print(f"  高波: {stab['regime_ic']['vol'].get('high_vol', {})}")
print(f"  低波: {stab['regime_ic']['vol'].get('low_vol', {})}")
print(f"\nIC 最大回撤: {stab['ic_max_drawdown']:.4f}")

### 5.4 尾部分析

极端信号的质量往往决定了实际交易收益。

In [ ]:
tail = tail_analysis(factor_panel, price_panel, threshold=0.9, horizon=5)
print(f"条件 IC: {tail['conditional_ic']:.4f}")
print(f"命中率: {tail['tail_hit_rate']:.1%}")
print(f"MAE (最大浮亏): {tail['mae']:.6f}")
print(f"尾部频率: {tail['tail_frequency']:.1%}")
print(f"尾部观测数: {tail['n_tail_observations']}")

## 5.5 参数扫描 — FamilyAnalyzer

当因子有多个参数变体时（如不同 lookback），使用 `FamilyAnalyzer` 进行参数空间扫描，
快速找到最优参数组合。

工作流: `sweep()` → `plot_sensitivity()` → `select()` → `detail()`

In [ ]:
# === FamilyAnalyzer 参数扫描示例 ===
# 需要已定义带 _param_grid 的因子类，以下使用 MultiScaleReturns 演示
#
# from factor_research.evaluation.family_analyzer import FamilyAnalyzer
# from factor_research.factors.momentum.returns import MultiScaleReturns
# from factor_research.core.types import DataType
#
# # 准备时序因子所需的数据格式: {symbol: {DataType: DataFrame}}
# # （需要 OHLCV 数据，此处假设已从 DataReader 获取）
# # data = {sym: {DataType.OHLCV: ohlcv_df} for sym, ohlcv_df in ...}
#
# fa = FamilyAnalyzer(
#     factor_class=MultiScaleReturns,   # 带 _param_grid 的因子类
#     data=data,                         # {symbol: {DataType: DataFrame}}
#     price_panel=price_panel,           # timestamp × symbol 价格面板
#     horizons=[1, 5, 10, 30],           # 评价 horizon
# )
#
# # Step 1: sweep — 轻量扫描全参数空间
# sweep_df = fa.sweep()
# print(sweep_df)  # 每行 = 一组参数 × 一个 horizon，列含 ic_mean, ic_ir 等
#
# # Step 2: plot — 可视化参数敏感度
# fig = fa.plot_sensitivity(metric="ic_ir")  # 折线图: 参数 vs 指标
# plt.show()
#
# # Step 3: select — 按阈值筛选最优参数
# best = fa.select(min_ic_ir=0.5, top_n=3, horizon=1, sort_by="ic_ir")
# print(best)
#
# # Step 4: detail — 对选定参数做完整 FactorAnalyzer 报告
# report = fa.detail(lookback=10)  # 传入具体参数值
# print(report.keys())  # ic, quantile, tail, stability, nonlinear, turnover
print("FamilyAnalyzer 示例代码已注释，取消注释后执行")

## 6. 因子代码化模板

验证完因子逻辑后，将其转化为标准的因子类，方便复用和管理。

In [ ]:
# === 因子代码化模板（复制到 factor_research/factors/ 目录下）===
FACTOR_TEMPLATE = '''
import pandas as pd
from factor_research.core.base import TimeSeriesFactor
from factor_research.core.registry import register_factor
from factor_research.core.types import DataRequest, DataType, FactorMeta, FactorType


@register_factor
class MyCustomFactor(TimeSeriesFactor):
    """自定义因子说明"""

    def meta(self) -> FactorMeta:
        return FactorMeta(
            name="my_custom_factor",
            display_name="自定义因子",
            factor_type=FactorType.TIME_SERIES,
            category="custom",
            description="因子的经济含义描述",
            data_requirements=[
                DataRequest(DataType.OHLCV, timeframe="1m", lookback_bars=5),
            ],
            output_freq="1m",
            version="1.0",
        )

    def compute_single(self, symbol: str, data: dict) -> pd.Series:
        ohlcv = data[DataType.OHLCV]
        if ohlcv.empty:
            return pd.Series(dtype=float)

        close = ohlcv["close"]
        # === 在此编写因子逻辑 ===
        result = close.pct_change(5)
        # === 因子逻辑结束 ===

        if "timestamp" in ohlcv.columns:
            result.index = pd.to_datetime(ohlcv["timestamp"], utc=True)
        result.name = self.meta().name
        return result.dropna()
'''
print(FACTOR_TEMPLATE)

## 7. 存储入库

将评价通过的因子存入 FactorStore，供 Phase 2b 使用。

In [ ]:
# === 因子入库 ===
# store = FactorStore()  # 默认路径: db/factors/
#
# meta = FactorMeta(
#     name="momentum_5m",
#     display_name="5分钟动量",
#     factor_type=FactorType.TIME_SERIES,
#     category="momentum",
#     description="过去5根1m K线的累计收益率",
#     data_requirements=[DataRequest(DataType.OHLCV, timeframe="1m")],
#     output_freq="1m",
#     version="1.0",
# )
# store.save("momentum_5m", factor_panel, meta)
# print(f"因子已入库: {store.exists('momentum_5m')}")
print("因子入库代码已注释，取消注释后执行")